# Assignment 3 Report - Residual Neural Network for classifying BloodMNIST images

### Tevyn Vergara | 46421201 | Electrical & Biomedical Engineer

#### DISCLAIMER: GPT-5.5 was used for this assignment for:
1. Layout tips
2. Syntax checking
3. Trouble shooting and debugging
4. Code and theory explanation
5. Re-organizing to look more modular and readable

In reference: [1]

#### References:

[1] OpenAI, “ChatGPT, GPT-5.5,” ChatGPT, May 18, 2026. [Online]. Available: https://chatgpt.com/

[2] D. P. Kingma and J. Ba, “Adam: A method for stochastic optimization,” arXiv preprint arXiv:1412.6980, 2014. doi: 10.48550/arXiv.1412.6980.

[3] J. Yang, R. Shi, D. Wei, Z. Liu, L. Zhao, B. Ke, H. Pfister, and B. Ni, “MedMNIST v2—A large-scale lightweight benchmark for 2D and 3D biomedical image classification,” Scientific Data, vol. 10, no. 1, Art. no. 41, Jan. 2023, doi: 10.1038/s41597-022-01721-8

## 1. Methodology

This report documents the development of a residual neural network for multi-class classification of BloodMNIST microscopic blood-cell images. The overall workflow is:

1. Load the official BloodMNIST train, validation, and test splits using MedMNIST.
2. Implement the required ResNet18-style architecture in PyTorch.
3. Train candidate models on the training set using cross-entropy loss and the Adam optimizer.
4. Use the validation set to tune channel numbers, learning rate, and number of epochs.
5. Select the best-performing model using validation performance.
6. Evaluate the selected final model on the held-out test set.
7. Compare the achieved performance with the MedMNIST reference experiment and discuss possible reasons for differences.

The validation set is used for model selection so that the test set remains an unbiased final performance estimate.


## 2. Architecture of the ResNet18 Neural Network

The implemented network follows a ResNet18-style residual architecture designed for BloodMNIST images with shape `3 x 28 x 28`. The network accepts RGB image tensors in PyTorch format `[batch_size, 3, 28, 28]`.

The architecture contains:

1. An initial convolutional feature extractor:
   `Conv2d(7x7, stride=2, padding=3) -> BatchNorm2d -> ReLU -> MaxPool2d(7x7, stride=2, padding=1)`.
2. A residual block cascade containing 8 residual blocks:
   Group 1 uses two Residual Block-I modules with `C1` channels.
   Group 2 uses one Residual Block-II followed by one Residual Block-I with `C2` channels.
   Group 3 uses one Residual Block-II followed by one Residual Block-I with `C3` channels.
   Group 4 uses one Residual Block-II followed by one Residual Block-I with `C4` channels.
3. A 2D global average pooling layer using `AdaptiveAvgPool2d((1, 1))`.
4. A flatten layer converting `[batch_size, C4, 1, 1]` to `[batch_size, C4]`.
5. A fully connected classifier:
   `Linear(C4, C4/2) -> ReLU -> Linear(C4/2, 8)`.

The output of the final layer is a vector of 8 raw logits, one for each BloodMNIST class. A softmax layer is not included in the model because `nn.CrossEntropyLoss` expects raw logits during training.

**Figure placeholder:** Insert architecture diagram or block summary here.

*Figure 1. Implemented ResNet18-style architecture for BloodMNIST classification.*


## 3. Dataset Loading

The dataset is loaded using the `load_bloodmnist_data(batch_size, download, size)` function in `A3code.py`. The function uses the MedMNIST metadata dictionary `INFO['bloodmnist']` to obtain the correct Python dataset class, number of channels, number of classes, and label information.

BloodMNIST contains 17,092 images split into:

- 11,959 training images
- 1,712 validation images
- 3,421 test images

Each image is a `28 x 28` RGB image with 3 colour channels. The function applies the following preprocessing pipeline:

```python
transforms.ToTensor()
transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
```

`ToTensor()` converts images into PyTorch tensor format `[channels, height, width]`, which is required by `nn.Conv2d`. The normalization step maps pixel values from approximately `[0, 1]` to approximately `[-1, 1]`, which helps stabilize neural network training.

The function loads the official MedMNIST splits using:

```python
DataClass(split='train', transform=data_transform, download=download, size=size)
DataClass(split='val', transform=data_transform, download=download, size=size)
DataClass(split='test', transform=data_transform, download=download, size=size)
```

The loaded datasets are wrapped in PyTorch `DataLoader` objects. The training dataloader uses `shuffle=True` so that batches are presented in a different order each epoch. The validation and test dataloaders use `shuffle=False` so that evaluation remains deterministic and predictions can be matched back to samples if required. When CUDA is available, `pin_memory=True` is used to improve transfer efficiency between CPU memory and GPU memory.


## 4. Training Procedure and Loss Curves

This section will describe the training loop used for each model. The model is trained using the training dataloader, and validation loss is computed after each epoch using the validation dataloader. The expected training procedure is:

1. Move image and label batches to the selected device (`cuda`, `mps`, or `cpu`).
2. Run the forward pass through the ResNet18 model.
3. Compute multi-class classification loss using `nn.CrossEntropyLoss`.
4. Backpropagate gradients with `loss.backward()`.
5. Apply gradient clipping using `torch.nn.utils.clip_grad_norm_`.
6. Update model weights using the Adam optimizer.
7. Record training loss and validation loss for each epoch.

The training and validation loss curves are important because they show whether the model is learning, underfitting, or overfitting.

**Figure placeholder:** Insert training and validation loss curve for Experiment 1 here.

*Figure 2. Training and validation loss per epoch for Experiment 1.*

**Figure placeholder:** Insert training and validation loss curve for Experiment 2 here.

*Figure 3. Training and validation loss per epoch for Experiment 2.*

**Figure placeholder:** Insert training and validation loss curve for Experiment 3 here.

*Figure 4. Training and validation loss per epoch for Experiment 3.*


## 5. Hyperparameter Selection Procedure

The three main hyperparameters tuned in this assignment are:

- Channel numbers: `[C1, C2, C3, C4]`
- Learning rate
- Maximum number of training epochs

The validation set is used to compare candidate models and select the best hyperparameter configuration. The test set is not used during this selection process.

Candidate channel settings to evaluate include:

```python
[8, 16, 32, 64]
[16, 32, 64, 128]
[32, 64, 128, 256]
[64, 128, 256, 512]
```

Candidate learning rates include:

```python
1e-3
1e-4
1e-5
```

The final number of epochs should be selected by observing validation performance and loss curves. A model that continues improving on training loss but stops improving on validation loss may be overfitting.

**Table placeholder:** Insert hyperparameter experiment summary table here.

| Experiment | Channel Numbers | Learning Rate | Max Epochs | Best Validation Accuracy | Best Validation Loss | Notes |
|---|---:|---:|---:|---:|---:|---|
| 1 | TBD | TBD | TBD | TBD | TBD | TBD |
| 2 | TBD | TBD | TBD | TBD | TBD | TBD |
| 3 | TBD | TBD | TBD | TBD | TBD | TBD |

**Figure placeholder:** Insert validation accuracy comparison graph here.

*Figure 5. Validation accuracy comparison across hyperparameter experiments.*


## 6. Final Model Performance Evaluation

After selecting the best model using validation performance, the final model is evaluated on the held-out test set. This section will report the final classification performance using metrics such as:

- Test accuracy
- Test loss
- Confusion matrix
- Per-class precision, recall, and F1-score if appropriate

The confusion matrix is useful because it shows which blood-cell classes are most often confused by the model.

**Table placeholder:** Insert final test metrics here.

| Metric | Value |
|---|---:|
| Test Loss | TBD |
| Test Accuracy | TBD |
| Macro Precision | TBD |
| Macro Recall | TBD |
| Macro F1-score | TBD |

**Figure placeholder:** Insert final model confusion matrix here.

*Figure 6. Confusion matrix for the selected final model on the BloodMNIST test set.*


## 7. Comparison of Other Model Performances

This section will compare the performance of models trained with different hyperparameter settings. The discussion should explain why certain configurations performed better or worse.

Possible discussion points include:

- Smaller channel settings may train faster but may not have enough capacity to represent complex blood-cell features.
- Larger channel settings may improve accuracy but require more GPU memory and may overfit if validation loss increases.
- A learning rate that is too high may cause unstable training or poor convergence.
- A learning rate that is too low may train too slowly or fail to reach good validation performance within the selected epoch limit.
- Increasing epochs can improve performance until validation loss plateaus or begins to worsen.

**Table placeholder:** Insert comparison of all trained models here.

| Model | Channel Numbers | Learning Rate | Epochs | Train Accuracy | Validation Accuracy | Test Accuracy | Training Time | Discussion Summary |
|---|---:|---:|---:|---:|---:|---:|---:|---|
| Model 1 | TBD | TBD | TBD | TBD | TBD | TBD | TBD | TBD |
| Model 2 | TBD | TBD | TBD | TBD | TBD | TBD | TBD | TBD |
| Model 3 | TBD | TBD | TBD | TBD | TBD | TBD | TBD | TBD |

**Figure placeholder:** Insert model performance comparison graph here.

*Figure 7. Performance comparison between trained models with different hyperparameters.*


## 8. Comparison with the MedMNIST Experiment

This section compares the best achieved BloodMNIST test performance with the performance reported in the MedMNIST reference experiment [3]. The comparison should use the same dataset split and should clearly state the metric being compared.

The discussion should explain possible reasons for any difference between this implementation and the reported MedMNIST result. Possible reasons include:

- Differences in model architecture or training details
- Different data augmentation or preprocessing choices
- Different optimizer settings, learning rate schedules, or epoch counts
- Random initialization and stochastic mini-batch training effects
- Hardware or runtime constraints affecting the amount of tuning performed

**Table placeholder:** Insert comparison with MedMNIST reference result here.

| Source | Model / Method | Test Accuracy | AUC | Notes |
|---|---|---:|---:|---|
| MedMNIST reference [3] | TBD | TBD | TBD | Reported benchmark |
| This work | Best ResNet18 model | TBD | TBD | Trained from scratch |

**Figure placeholder:** Insert benchmark comparison figure here if useful.

*Figure 8. Comparison between the selected model and the MedMNIST reported performance.*


## 9. Model Loading and Reproducibility Notes

This section will explain how the saved trained models can be loaded and evaluated. This is required so that the reported results can be reproduced by assessors.

Each saved model checkpoint should include enough information to reconstruct the architecture and evaluation conditions, such as:

```python
channel_nums
learning_rate
epoch_num
model_state_dict
optimizer_state_dict
best_validation_accuracy
best_validation_loss
test_accuracy
```

The final code should include an evaluation section that:

1. Recreates the ResNet18 architecture using the saved `channel_nums` and number of classes.
2. Loads the saved `model_state_dict`.
3. Moves the model to the available device.
4. Runs the model on the BloodMNIST test dataloader.
5. Prints or saves the same metrics reported in this notebook.

**Link placeholder:** Insert OneDrive or Google Drive folder link containing trained models here.

**Timestamp placeholder:** Insert upload timestamp for submitted trained models here.


## 10. Possible Architecture Improvements

This section discusses possible ways to improve the model beyond the required architecture. Any optional improvement should still avoid pretrained models.

Possible improvements include:

- Adding data augmentation such as random rotation, horizontal/vertical flips, or colour jitter if biologically appropriate.
- Adding dropout in the classifier to reduce overfitting.
- Trying learning rate scheduling, such as reducing the learning rate when validation loss plateaus.
- Increasing channel sizes if GPU memory allows.
- Adjusting the initial convolution or pooling design for very small `28 x 28` images, since aggressive early downsampling may remove useful spatial detail.
- Testing different optimizer settings such as Adam weight decay.

If an improved architecture is tested, its results should be compared fairly against the required baseline using the same train, validation, and test splits.


# SCRATCH NOTES FOR ME. DO NOT PARSE/WORRY THIS PART 

ResNet18

Reference: https://medmnist.com/v2 

Accuracy: 0.998 (Look up formula)

Area Under Curve: 0.958


### QUESTIONS TO ASK TUTOR ON PRIVATE ###

1. Different training and validation techniques? Recommended ones besides CrossEntropyLoss()?
2. Recommended techniques for tuning channel numbers and learning rate? Would it then be optimal to tune epochs AFTER? (First channels, then learning rate, then epochs)?
3. Just one report that contains all the evaluation metrics of developed models?
4. Can I use PyTorch official ResNet18 as a benchmark comparison? (Using lit source) and talk about why their hyperparameters outperform my own?
5. During interview, do you want us to explain the code that is already there? (Residual Blocks) and other draft?
6. For different trained models, can I just copy/paste the raw code as seperate files? Or is there a better approach to help with your marking?
7. For overall model evaluation I wanted to use a confusion matrix to show the classification distributions of all my trained models but I wanted to focus on a "optimal" model where I wanted specifically to look at how accurate the model performed vs. how long it took to train. Would this be a good main discussion point to talk about?

To change:

1. In precision matrix figure, label the classes as their numerical label 0-8. Add a legend on the left hand size corner of the graph with the numerical labels and their actual sceintific labels. Also make the figure title, and the predicted and actual tables bold font. 

2. Make a seperate table that includes these metrics:
Final Test Accuracy (%): 
Final Test Loss: 
Final Test Macro Precision:
Final Test Weighted Precision:
Final Test Macro Recall:
Final Test Weighted Recall: 
Total Training Time (seconds): 
"Gap"
Lowest Training Loss:
Highest Training Accuracy:
Lowest Training Loss:
Highest Validation Accuracay:

3. Remove the confusion matrix out of the terminal output